# 模板字面量类型

学习目标：能构造受约束的字符串类型、从模板推断片段，并组合属性类型生成事件及访问器接口。

前置知识：字符串模板、字面量联合、泛型、条件类型与 infer、映射类型和键重映射。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ES 模块，开启 strict。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/15-template-literal-types/。

1. [main.ts](scripts/15-template-literal-types/main.ts)：配套实现与示例。
2. [tsconfig.json](scripts/15-template-literal-types/tsconfig.json)：本章独立项目配置。
3. [type-errors.ts](scripts/15-template-literal-types/type-errors.ts)、[tsconfig.errors.json](scripts/15-template-literal-types/tsconfig.errors.json)：单独检查的类型反例。

Step 1：检查本章正常示例的类型。

```bash
npm run check:15
```

Step 2：生成本章 JavaScript。

```bash
npm run build:15
```

Step 3：运行本章正常示例。

```bash
npm run run:15
# 正常退出；各段预期输出见代码注释。
```

正常片段均按正文顺序节选自 main.ts，前文定义在后续片段中继续使用。直接运行完整项目；反例使用独立配置，不进入正常运行入口。

## 1 拼接字符串类型与联合展开

多个插值位置各自是联合时，逐行、逐列组合比把长联合背下来更直观。

模板字面量类型（template literal type）采用字符串模板语法，但位于类型位置。模板插值中的类型若是联合，会展开各个成员；多个位置的联合进行组合。

下面两个领域乘以两个动作，得到四个合法名称。类型不会生成字符串值，也不会检查外部文本；value 仍需要真正写出或在运行时构造。

![模板中的两个联合形成四种组合。每个格子是一个允许的字符串字面量类型，不是生成的运行时数据。](image/illustration/15-01-template-literal-product.svg)

图示说明：依据模板联合组合规则自绘，本例只有两个领域和两个动作，因此共有四种字面量；不表示运行时已经存在四个字符串。

读下面 EventName 的定义时，先定位 value 所在格子，再尝试把动作改为表外名称，检查类型约束。

```typescript
export type Domain = "lesson" | "quiz";
type Action = "open" | "close";
export type EventName = `${Domain}:${Action}`;
const value: EventName = "lesson:open";
console.log(value);
// 预期输出：lesson:open
```

## 2 从模板中推断片段

条件类型可以使用 infer 匹配字符串字面量。S 代表待处理的字符串类型，Key 是移除 Changed 后缀后推断出的片段；没有后缀时得到 never。

这里只对已知类型做匹配。它不执行任意文本的运行时解析；本例直接使用已知字面量观察类型关系。

```typescript
export type ChangedKey<S extends string> = S extends `${infer Key}Changed` ? Key : never;
const key: ChangedKey<"titleChanged"> = "title";
console.log(key);
// 预期输出：title
```

## 3 事件名与回调参数对应

事件名只限制字符串还不够，回调也应收到相应属性的值。下面 K 代表 Model 的字符串属性键，泛型函数从事件名中的 K 推断具体键，再用 Model[K] 确定回调参数。

listener 返回一个交付值的函数：调用它时才把值送给回调。它没有自动观察对象变化，事件名和值的联系由类型签名表达，真正交付由函数体完成。

```typescript
export type Model = { title: string; hours: number };
export function listener<K extends string & keyof Model>(event: `${K}Changed`, callback: (value: Model[K]) => void) {
  return (value: Model[K]) => {
    console.log(event);
    callback(value);
  };
}
const notifyHours = listener("hoursChanged", value => console.log(value.toFixed(1)));
notifyHours(3);
// 预期输出：hoursChanged
// 预期输出：3.0
```

## 4 字符串转换类型

下列工具处理字符串类型，不调用运行时转换函数；它们也不是依赖地区设置的自然语言大小写规则。S 表示输入的字符串类型，实际字面量要符合转换后的结果。

| 完整名称 | 中文名称／含义 | 处理范围 |
| --- | --- | --- |
| Uppercase&lt;S&gt; | 转为大写 | 整个字符串 |
| Lowercase&lt;S&gt; | 转为小写 | 整个字符串 |
| Capitalize&lt;S&gt; | 首字符大写 | 只改变首字符 |
| Uncapitalize&lt;S&gt; | 首字符小写 | 只改变首字符 |

```typescript
const upper: Uppercase<"lessonId"> = "LESSONID";
const lower: Lowercase<"LessonID"> = "lessonid";
const capital: Capitalize<"lessonID"> = "LessonID";
const uncapital: Uncapitalize<"LessonID"> = "lessonID";
console.log(upper, lower, capital, uncapital);
// 预期输出：LESSONID lessonid LessonID lessonID
```

## 5 模板与键重映射配合

Getters 中 T 是原对象类型，P 是正在处理的键。string 与 P 的交叉只留下能参与字符串改名的键，Capitalize 处理首字符，模板再加 get 前缀。

得到的只是访问器接口，不会自动生成访问器函数。下面手写两个实现，返回值仍分别受原属性类型限制；若对象含 symbol 键或数值键，string & P 会将这些键排除，不为它们生成字符串访问器。

代入 Model 后，title → Title → getTitle，返回值为 string；hours → Hours → getHours，返回值为 number。因此结果接口有 getTitle(): string 和 getHours(): number 两个方法。

```typescript
export type Getters<T> = {
  [P in keyof T as `get${Capitalize<string & P>}`]: () => T[P]
};
const getters: Getters<Model> = {
  getTitle: () => "模板",
  getHours: () => 3
};
console.log(getters.getTitle(), getters.getHours());
// 预期输出：模板 3
```

## 6 控制联合规模

模板各位置的选择会相乘。例如三个位置分别有 2、3、2 个互不冲突的候选，就有 12 种组合。候选增加时，编辑器和类型检查器需要处理的联合也会扩大。

小型事件表适合保留精确联合；大型自动生成的标识清单应考虑提前生成类型文件，或以运行时解析与校验表达规则。不要为了演示规模刻意生成巨大的类型，也不能用 string 偷换本来需要的有限键约束。

## 7 检查类型边界

下面的 [type-errors.ts](scripts/15-template-literal-types/type-errors.ts) 只用于检查，不执行。逐项阅读注释，修正时保留原本需求，不通过断言或关闭检查掩盖错误。

```typescript
import { listener, type EventName, type ChangedKey, type Getters, type Model } from "./main.js";
const wrong: EventName = "lesson:save"; // save 不在动作联合中。
const unmatched: ChangedKey<"hours"> = "hours"; // 模板不匹配，结果为 never。
listener("missingChanged", () => {}); // missing 不属于 Model 的字符串键。
listener("hoursChanged", value => value.toUpperCase()); // 回调值为 number。
const wrongGetter: Getters<Model> = { getTitle: () => "标题", getHours: () => "3" }; // 返回值应为 number。
// 预期诊断包含：TS2322, TS2345, TS2339。
```

Step 1：单独检查反例并对照错误位置与原因。

```bash
npm run errors:15
# 本章固定编译器预期退出码为 1；正常项目命令的退出码为 0。
```

## 本章小结

模板类型构造字符串约束，infer 可提取已知字面量片段，键重映射可派生接口；它们都不生成运行时解析或监听行为。控制联合规模能让类型保持易读、可检查。

## 练习

1. 增加动作 save，核对 lesson:save 和 quiz:save 被接受，未知领域被拒绝。

2. 为 Model 增加 active: boolean 并创建 activeChanged 的 listener；交付 true，核对回调能使用布尔值而不能调用数值方法。

3. 为 Model 的新属性实现 getActive，核对 Getters&lt;Model&gt; 不能漏掉该方法。

### 提示

1. 只扩展 Action，不扩展 Domain，合法组合变为六个。
2. 新增 active 后，既有 Getters&lt;Model&gt; 对象也需要新方法；与第 3 题一并补齐。
3. K 为 active 时回调值为 Model["active"]，P 为 active 时访问器名为 getActive。


### 参考解析

1. lesson:save、quiz:save 均符合模板；未知领域如 user:save 仍不属于 EventName。
2. listener("activeChanged", value =&gt; console.log(value)) 返回接收 boolean 的函数；交付 true 时依次输出 activeChanged 和 true，value.toFixed 则应诊断失败。
3. 添加 getActive: () =&gt; true；缺少这个必需方法时对象不满足 Getters&lt;Model&gt;，返回字符串也不满足 boolean 返回类型。


## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [Template Literal Types：展开、推断、字符串工具与规模](https://www.typescriptlang.org/docs/handbook/2/template-literal-types.html)；[Mapped Types：Key Remapping via as](https://www.typescriptlang.org/docs/handbook/2/mapped-types.html#key-remapping-via-as)；[4.1：Template Literal Types](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-4-1.html#template-literal-types)。 |
